# **Map Visual Function showcasing Capacity visuals across NYC**

## Static Visuals for Docked Systems

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple, Union

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.patches import Circle, FancyArrow, Rectangle


def plot_nyc_only_capacity_norm_tracts_with_compass_scale(
    *,
    capacity_csv: Union[str, Path],
    ny_tract_shp: Union[str, Path],
    output_dir: Union[str, Path],
    csv_tract_col: str = "census_tract",
    shp_geoid_col: str = "GEOID",
    # NYC borough prefixes ONLY
    nyc_borough_prefixes: Tuple[str, ...] = ("36061", "36047", "36081", "36005"),
    # gate column: only tracts with values here will be shown
    capacity_norm_col: str = "total_capacity_norm",
    # if True, remove capacity_norm == 0 too
        # PROF-style docked filtering (capacity-file driven)
    station_count_col: str = "num_station",   # column in capacity CSV that indicates station count
    min_stations: int = 1,                    # keep tracts with >= this many stations
    gate_col: Optional[str] = None,           # optional extra gate (defaults to capacity_norm_col)
    drop_zeros: bool = True,
    # which columns to map (will only include tracts that pass the gate)
    value_cols: Optional[Sequence[str]] = None,
    # plot styling
    figsize: Tuple[int, int] = (10, 10),
    cmap: str = "RdYlBu",
    edgecolor: str = "black",
    linewidth: float = 0.30,
    alpha: float = 0.90,
    legend_loc: str = "upper left",
    add_basemap: bool = True,
    # add-ons
    add_compass: bool = True,
    add_scalebar: bool = True,
    scalebar_segments_km: Tuple[int, int, int] = (0, 2, 5),
) -> Dict[str, Path]:
    """
    NYC-only tract maps (Manhattan/Brooklyn/Queens/Bronx) that show ONLY tracts where capacity_norm_col exists
    (and optionally > 0). Produces decile maps with:
      - OpenStreetMap basemap
      - Compass rose (N/E/S/W) in top-right
      - Scale bar (0-2-5 km style) in bottom-right
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # ----------------------------
    # Helpers
    # ----------------------------
    def to_geoid11(s: pd.Series) -> pd.Series:
        return (
            s.astype(str)
            .str.strip()
            .str.replace(r"\.0$", "", regex=True)
            .str.replace(r"\s+", "", regex=True)
            .str.zfill(11)
        )

    def add_decile_labels(gdf: gpd.GeoDataFrame, value_col: str, label_col: str) -> None:
        p = gdf[value_col].rank(pct=True, method="average")
        bins = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
        labels = [
            "1-10%", "11-20%", "21-30%", "31-40%", "41-50%",
            "51-60%", "61-70%", "71-80%", "81-90%", "91-100%"
        ]
        gdf[label_col] = pd.cut(p, bins=bins, labels=labels, include_lowest=True)

    def add_compass_rose(ax, xlim, ylim, frac_pos=(0.88, 0.88), size_frac=0.075):
        width = xlim[1] - xlim[0]
        height = ylim[1] - ylim[0]
        cx = xlim[0] + frac_pos[0] * width
        cy = ylim[0] + frac_pos[1] * height
        R = size_frac * min(width, height)

        outer = Circle((cx, cy), R, facecolor="white", edgecolor="black", linewidth=1.2, zorder=10)
        inner = Circle((cx, cy), R * 0.88, facecolor="white", edgecolor="black", linewidth=0.8, zorder=11)
        ax.add_patch(outer)
        ax.add_patch(inner)

        ax.text(cx, cy + R + 0.012 * height, "N", ha="center", va="bottom",
                fontsize=12, fontweight="bold", zorder=12)
        ax.text(cx, cy - R - 0.012 * height, "S", ha="center", va="top",
                fontsize=10, zorder=12)
        ax.text(cx - R - 0.006 * width, cy, "W", ha="right", va="center",
                fontsize=10, zorder=12)
        ax.text(cx + R + 0.006 * width, cy, "E", ha="left", va="center",
                fontsize=10, zorder=12)

        ax.add_patch(
            FancyArrow(
                cx, cy - 0.18 * R, 0, 0.72 * R,
                width=0.0,
                head_width=0.22 * R,
                head_length=0.28 * R,
                length_includes_head=True,
                facecolor="black",
                edgecolor="black",
                zorder=13,
            )
        )

    def add_scalebar_km(ax, xlim, ylim, frac_pos=(0.72, 0.06), bar_height_frac=0.012, segments_km=(0, 2, 5)):
        # EPSG:3857 => meters
        width = xlim[1] - xlim[0]
        height = ylim[1] - ylim[0]
        x0 = xlim[0] + frac_pos[0] * width
        y0 = ylim[0] + frac_pos[1] * height

        total_m = segments_km[-1] * 1000.0
        bar_h = bar_height_frac * height

        seg1_m = (segments_km[1] - segments_km[0]) * 1000.0
        seg2_m = (segments_km[2] - segments_km[1]) * 1000.0

        r1 = Rectangle((x0, y0), seg1_m, bar_h, facecolor="black", edgecolor="black", linewidth=0.8, zorder=10)
        r2 = Rectangle((x0 + seg1_m, y0), seg2_m, bar_h, facecolor="white", edgecolor="black", linewidth=0.8, zorder=10)
        ax.add_patch(r1)
        ax.add_patch(r2)

        ax.plot([x0, x0], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)
        ax.plot([x0 + seg1_m, x0 + seg1_m], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)
        ax.plot([x0 + total_m, x0 + total_m], [y0, y0 + bar_h], color="black", linewidth=1.0, zorder=11)

        ax.text(x0, y0 - 0.012 * height, f"{segments_km[0]}", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + seg1_m, y0 - 0.012 * height, f"{segments_km[1]} km", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + total_m, y0 - 0.012 * height, f"{segments_km[2]} km", ha="center", va="top", fontsize=10, zorder=12)
        ax.text(x0 + total_m / 2, y0 + bar_h + 0.008 * height, "Scale", ha="center", va="bottom", fontsize=10, zorder=12)

    # ----------------------------
    # 1) Load NY tracts + filter to NYC boroughs FIRST
    # ----------------------------
    tracts = gpd.read_file(ny_tract_shp)
    if shp_geoid_col not in tracts.columns:
        raise KeyError(f"Tract shapefile missing '{shp_geoid_col}'. Columns: {list(tracts.columns)}")

    tracts[shp_geoid_col] = to_geoid11(tracts[shp_geoid_col])
    tracts_nyc = tracts[tracts[shp_geoid_col].str.startswith(nyc_borough_prefixes)].copy()
    if tracts_nyc.empty:
        raise ValueError("NYC borough filter returned 0 tracts. Check GEOID formatting or prefixes.")

        # ----------------------------
    # 2) Load capacity CSV + PROF-style docked tract universe
    #    (capacity-file driven, but uses station presence)
    # ----------------------------
    cap_raw = pd.read_csv(capacity_csv)
    if csv_tract_col not in cap_raw.columns:
        raise KeyError(f"capacity_csv missing '{csv_tract_col}'")

    cap_raw[csv_tract_col] = to_geoid11(cap_raw[csv_tract_col])

    # professor-style docked needs station_count_col
    if station_count_col not in cap_raw.columns:
        raise KeyError(
            f"capacity_csv missing '{station_count_col}'. "
            f"Available columns: {list(cap_raw.columns)}"
        )

    # if gate_col not provided, use capacity_norm_col (your current behavior) BUT applied later
    if gate_col is None:
        gate_col = capacity_norm_col

    if gate_col not in cap_raw.columns:
        raise KeyError(f"'{gate_col}' not found in capacity_csv. Available: {list(cap_raw.columns)}")

    # --- IMPORTANT: aggregate per tract (prof notebooks often do this implicitly)
    # Choose sane aggregation:
    # - capacities => SUM
    # - rates/ratios => MEAN
    sum_cols = [c for c in cap_raw.columns if "capacity" in c and c != csv_tract_col]
    mean_cols = [c for c in cap_raw.columns if c in {"occupancy_rate", "return_pressure"}]

    agg = {}
    for c in sum_cols:
        if c in cap_raw.columns:
            agg[c] = "sum"
    for c in mean_cols:
        if c in cap_raw.columns:
            agg[c] = "mean"

    # Make sure station count aggregates properly (sum is fine if file has multiple rows/tract)
    agg[station_count_col] = "sum"

    # Keep any other numeric columns not covered (use mean as a safe default)
    numeric_cols = cap_raw.select_dtypes(include="number").columns.tolist()
    for c in numeric_cols:
        if c in (station_count_col,) or c in agg:
            continue
        agg[c] = "mean"

    cap = cap_raw.groupby(csv_tract_col, as_index=False).agg(agg)

    # --- PROF-style docked tract universe: tracts with stations
    tracts_with_stations = cap.loc[cap[station_count_col].fillna(0) >= min_stations, csv_tract_col].unique()

    if len(tracts_with_stations) == 0:
        raise ValueError(
            f"0 tracts found with {station_count_col} >= {min_stations}. "
            f"Check your station_count_col or the capacity file."
        )

    # ----------------------------
    # 3) Filter NYC tracts to those with stations (universe first), then merge
    # ----------------------------
    tracts_nyc = tracts_nyc[tracts_nyc[shp_geoid_col].isin(tracts_with_stations)].copy()
    if tracts_nyc.empty:
        raise ValueError(
            "After applying prof-style docked filter (tracts with stations), 0 NYC tracts remain. "
            "This usually means census_tract GEOIDs don't match shapefile GEOIDs."
        )

    # Merge capacity metrics onto ONLY those tracts
    # left merge keeps geometry stable even if a metric column is missing for a tract
    gdf = tracts_nyc.merge(cap, left_on=shp_geoid_col, right_on=csv_tract_col, how="left")

    # Now apply your old gate logic *after* universe alignment + aggregation:
    # - Require the gate_col to be present
    gdf = gdf.dropna(subset=[gate_col]).copy()

    # - Optionally drop zeros (still supported)
    if drop_zeros:
        gdf = gdf[gdf[gate_col] > 0].copy()

    if gdf.empty:
        raise ValueError(
            f"0 rows remain after prof-style filtering + gating on '{gate_col}' "
            f"(drop_zeros={drop_zeros}). Try drop_zeros=False to confirm."
        )


    # ----------------------------
    # 4) Choose plot columns
    # ----------------------------
    if value_cols is None:
        value_cols = [
            "total_capacity_norm",
            "vehicle_capacity_norm",
            "dock_capacity_norm",
            "occupancy_rate",
            "return_pressure",
        ]

    missing = [c for c in value_cols if c not in gdf.columns]
    if missing:
        raise KeyError(f"Missing plot columns in merged gdf: {missing}")

    # ----------------------------
    # 5) Plot decile choropleths + basemap + compass + scalebar
    # ----------------------------
    gdf_web = gdf.to_crs(epsg=3857)
    xmin, ymin, xmax, ymax = gdf_web.total_bounds
    xlim = (xmin, xmax)
    ylim = (ymin, ymax)

    saved: Dict[str, Path] = {}

    for col in value_cols:
        tmp = gdf_web.dropna(subset=[col]).copy()
        if tmp.empty:
            continue

        decile_col = f"{col}_decile"
        add_decile_labels(tmp, col, decile_col)

        fig, ax = plt.subplots(figsize=figsize)

        tmp.plot(
            ax=ax,
            column=decile_col,
            categorical=True,
            legend=True,
            cmap=cmap,
            edgecolor=edgecolor,
            linewidth=linewidth,
            alpha=alpha,
            legend_kwds={"loc": legend_loc},
        )

        # lock extent BEFORE basemap
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)

        if add_basemap:
            ctx.add_basemap(ax, source=ctx.providers.OpenStreetMap.Mapnik)
            # lock extent AFTER basemap
            ax.set_xlim(xlim)
            ax.set_ylim(ylim)

        # add overlays on top of everything
        if add_compass:
            add_compass_rose(ax, xlim, ylim, frac_pos=(0.88, 0.88), size_frac=0.075)

        if add_scalebar:
            add_scalebar_km(ax, xlim, ylim, frac_pos=(0.72, 0.06), segments_km=scalebar_segments_km)

        ax.set_title(col.replace("_", " ").title(), fontsize=15, fontweight="bold")
        ax.axis("off")

        out = Path(output_dir) / f"NYC_ONLY_{capacity_norm_col}_gate__{col}.png"
        plt.savefig(out, dpi=300, bbox_inches="tight")
        plt.close(fig)

        saved[col] = out

    if not saved:
        raise ValueError("No maps saved (all requested columns were empty after filtering).")

    return saved


In [ ]:
saved = plot_nyc_only_capacity_norm_tracts_with_compass_scale(
    capacity_csv=r"D:\Research Fellowship\NYC_Docked_Output_v2\capacity_tract_with_vehicle_and_docks_norm.csv",
    ny_tract_shp=r"D:\Research Fellowship\Summer Research Stuff\Clean_Utilities\Capacity\NYC\tl_2024_36_tract.shp",
    output_dir=r"D:\Research Fellowship\NYC_Docked_Output_v2\STATIC_TRACT_MAPS_OUT",
    capacity_norm_col="total_capacity_norm",
    station_count_col="num_station",   
    min_stations=1,                    # <-- keep tracts with >=1 station
    drop_zeros=True,
    add_compass=True,
    add_scalebar=True,
    scalebar_segments_km=(0, 2, 5),
)
print(saved)


{'total_capacity_norm': WindowsPath('D:/Research Fellowship/NYC_Docked_Output_v2/STATIC_TRACT_MAPS_OUT/NYC_ONLY_total_capacity_norm_gate__total_capacity_norm.png'), 'vehicle_capacity_norm': WindowsPath('D:/Research Fellowship/NYC_Docked_Output_v2/STATIC_TRACT_MAPS_OUT/NYC_ONLY_total_capacity_norm_gate__vehicle_capacity_norm.png'), 'dock_capacity_norm': WindowsPath('D:/Research Fellowship/NYC_Docked_Output_v2/STATIC_TRACT_MAPS_OUT/NYC_ONLY_total_capacity_norm_gate__dock_capacity_norm.png'), 'occupancy_rate': WindowsPath('D:/Research Fellowship/NYC_Docked_Output_v2/STATIC_TRACT_MAPS_OUT/NYC_ONLY_total_capacity_norm_gate__occupancy_rate.png'), 'return_pressure': WindowsPath('D:/Research Fellowship/NYC_Docked_Output_v2/STATIC_TRACT_MAPS_OUT/NYC_ONLY_total_capacity_norm_gate__return_pressure.png')}


: 